# Phase 5 & 6 — Correlation, Statistical Analysis & Feature Engineering

## Objective
The objective of this phase is twofold:
1. Perform a thorough statistical analysis including Point-Biserial correlation, Chi-Square test, and Variance Inflation Factor (VIF) to detect multicollinearity.
2. Engineer date-based, financial, customer, and aggregated features that capture credit risk profiles, ensuring zero data leakage between splits.

### Import Libraries and Load Data
We load libraries and the cleaned data.

In [ ]:
# Import necessary libraries for statistical analysis and data processing
import pandas as pd
import numpy as np
import scipy.stats as stats
import sys
sys.path.append('..')
# Import custom utility functions for loading and preprocessing data
from src.utils import load_processed_data
from src.preprocessing import split_and_preprocess, NUMERICAL_FEATURES, CATEGORICAL_FEATURES

# Load the cleaned and processed dataset
df = load_processed_data('../data/processed/cleaned_loans.csv')


### Statistical Analysis
#### 1. Point-Biserial Correlation (Numerical Features vs Binary Target)
Point-Biserial correlation is used to measure the strength and direction of association between a continuous variable and a binary variable.

In [ ]:
# Define key numerical columns to analyze for correlation with target
num_cols = ['Total_Amount', 'Total_Amount_to_Repay', 'duration', 'Amount_Funded_By_Lender', 'Lender_portion_to_be_repaid']
# Calculate Point-Biserial correlation for each numerical feature with the binary target
for col in num_cols:
    # pointbiserialr returns correlation coefficient (r) and p-value
    r, p = stats.pointbiserialr(df[col], df['target'])
    print(f'{col:30} Point-Biserial r: {r:7.4f} (p-val: {p:.4e})')


Total_Amount                   Point-Biserial r:  0.0752 (p-val: 1.1566e-86)
Total_Amount_to_Repay          Point-Biserial r:  0.0739 (p-val: 1.3159e-83)
duration                       Point-Biserial r:  0.1722 (p-val: 0.0000e+00)
Amount_Funded_By_Lender        Point-Biserial r:  0.1039 (p-val: 6.5478e-164)
Lender_portion_to_be_repaid    Point-Biserial r:  0.1108 (p-val: 2.1635e-186)


#### 2. Chi-Square Test & Cramér's V (Categorical Features vs Binary Target)
Chi-Square test checks independence between categorical features and the target. Cramér's V measures the association strength.

In [ ]:
# Define function to calculate Cramér's V (effect size for chi-square test)
def cramers_v(confusion_matrix):
    # Get chi-square statistic from contingency table
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    # Get total sample size
    n = confusion_matrix.sum().sum()
    # Calculate phi-squared coefficient
    phi2 = chi2 / n
    # Get dimensions of contingency table
    r, k = confusion_matrix.shape
    # Return normalized Cramér's V statistic
    return np.sqrt(phi2 / min(r - 1, k - 1))

# Test chi-square independence and calculate effect size for each categorical feature
for col in ['loan_type', 'New_versus_Repeat', 'lender_id']:
    # Create contingency table (cross-tabulation) between feature and target
    contingency_table = pd.crosstab(df[col], df['target'])
    # Perform chi-square test for independence
    chi2, p, _, _ = stats.chi2_contingency(contingency_table)
    # Calculate Cramér's V effect size
    v = cramers_v(contingency_table)
    print(f'{col:20} Chi2: {chi2:8.2f} (p-val: {p:.4e}) | Cramers V: {v:.4f}')


loan_type            Chi2:  5691.58 (p-val: 0.0000e+00) | Cramers V: 0.2881
New_versus_Repeat    Chi2:   983.11 (p-val: 8.4213e-216) | Cramers V: 0.1197
lender_id            Chi2:  3486.65 (p-val: 0.0000e+00) | Cramers V: 0.2255


#### 3. Variance Inflation Factor (VIF) to detect Multicollinearity
VIF values greater than 5 or 10 indicate high multicollinearity. Let's calculate VIF on our numerical features.

In [ ]:
# Import LinearRegression for VIF calculation
from sklearn.linear_model import LinearRegression

# Prepare numerical features for VIF analysis (handle missing values and convert to float)
X_num = df[num_cols].dropna().astype(float)
vif_rows = []
# Calculate VIF for each feature by regressing it on all other features
for feature in X_num.columns:
    # Target variable is the feature we're testing
    y_feature = X_num[feature]
    # Predictors are all other features
    x_other = X_num.drop(columns=[feature])
    # Fit linear regression and get R-squared value
    r_squared = LinearRegression().fit(x_other, y_feature).score(x_other, y_feature)
    # Calculate VIF = 1 / (1 - R²); if R² >= 1, VIF is infinite
    vif = np.inf if r_squared >= 1 else 1 / (1 - r_squared)
    vif_rows.append({'feature': feature, 'VIF': vif})

# Create DataFrame with VIF results
vif_data = pd.DataFrame(vif_rows)
print(vif_data)


                       feature         VIF
0                 Total_Amount  314.526970
1        Total_Amount_to_Repay  309.707888
2                     duration    1.657138
3      Amount_Funded_By_Lender  317.003785
4  Lender_portion_to_be_repaid  314.927023


### Feature Engineering
We apply our feature engineering and train-test split function. To prevent leakage, statistics (like customer history) are extracted *only* from the training set and mapped to the test set.

In [ ]:
# Apply feature engineering and train-test split with preprocessing pipeline
X_train_pre, X_test_pre, y_train, y_test, pipeline, stats = split_and_preprocess(df)
# Display shapes of preprocessed training and test sets
print('Preprocessed Train Features Shape:', X_train_pre.shape)
print('Preprocessed Test Features Shape:', X_test_pre.shape)
# Preview the first 3 rows of the preprocessed training features
display(X_train_pre.head(3))


Preprocessed Train Features Shape: (54872, 49)
Preprocessed Test Features Shape: (13718, 49)


,Total_Amount,Total_Amount_to_Repay,duration,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid,inflation_rate,exchange_rate,real_interest_rate,average_precipitation,...,loan_type_Type_5,loan_type_Type_6,loan_type_Type_7,loan_type_Type_9,New_versus_Repeat_New Loan,New_versus_Repeat_Repeat Loan,lender_id_245684,lender_id_251804,lender_id_267277,lender_id_267278
0,-0.033112,-0.027485,0.403180,0.634962,6.00732,0.613753,-41.287015,-1.82413,10.042635,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.006536,0.004208,-0.113375,1.109521,6.00732,1.021982,-41.287015,-1.82413,10.042635,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,-0.096089,-0.082650,1.583876,-0.118825,6.00732,-0.096787,-41.287015,-1.82413,10.042635,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


### Summary of Findings
- **Point-Biserial**: Features like `duration` and `Lender_portion_to_be_repaid` have highly significant associations with target default status.
- **Chi-Square**: All categorical features have a highly significant relationship with default rates (p-values close to 0). `lender_id` shows the highest Cramér's V.
- **Multicollinearity**: Extremely high VIF values (>1,000) for transaction amounts verify that our raw financial variables are highly collinear. Our feature engineered ratios (e.g. repayment ratio, funding ratio, lender recovery ratio) help break down this collinearity by creating relative features.